In [14]:
import pandas as pd
print("Hello, World!")

Hello, World!


In [15]:
# Load the data
df = pd.read_csv('../data/superStore_sales_dataset.csv')
df.head()

,Row ID+O6G3A1:R6,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Category,Sub-Category,Product Name,Sales,Quantity,Profit,Returns,Payment Mode,ind1,ind2
0,4918,CA-2019-160304,01-01-2019,07-01-2019,Standard Class,BM-11575,Brendan Murry,Corporate,United States,Gaithersburg,...,Furniture,Bookcases,"Bush Westfield Collection Bookcases, Medium Ch...",73.94,1,28.2668,NaN,Online,NaN,NaN
1,4919,CA-2019-160304,02-01-2019,07-01-2019,Standard Class,BM-11575,Brendan Murry,Corporate,United States,Gaithersburg,...,Furniture,Bookcases,"Bush Westfield Collection Bookcases, Medium Ch...",173.94,3,38.2668,NaN,Online,NaN,NaN
2,4920,CA-2019-160304,02-01-2019,07-01-2019,Standard Class,BM-11575,Brendan Murry,Corporate,United States,Gaithersburg,...,Technology,Phones,GE 30522EE2,231.98,2,67.2742,NaN,Cards,NaN,NaN
3,3074,CA-2019-125206,03-01-2019,05-01-2019,First Class,LR-16915,Lena Radford,Consumer,United States,Los Angeles,...,Office Supplies,Storage,Recycled Steel Personal File for Hanging File ...,114.46,2,28.6150,NaN,Online,NaN,NaN
4,8604,US-2019-116365,03-01-2019,08-01-2019,Standard Class,CA-12310,Christine Abelman,Corporate,United States,San Antonio,...,Technology,Accessories,Imation Clip USB flash drive - 8 GB,30.08,2,-5.2640,NaN,Online,NaN,NaN


In [16]:
# Display the shape of the DataFrame
df.shape

(5901, 23)

In [17]:
# Display the summary information about the DataFrame
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 5901 entries, 0 to 5900
Data columns (total 23 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Row ID+O6G3A1:R6  5901 non-null   int64  
 1   Order ID          5901 non-null   str    
 2   Order Date        5901 non-null   str    
 3   Ship Date         5901 non-null   str    
 4   Ship Mode         5901 non-null   str    
 5   Customer ID       5901 non-null   str    
 6   Customer Name     5901 non-null   str    
 7   Segment           5901 non-null   str    
 8   Country           5901 non-null   str    
 9   City              5901 non-null   str    
 10  State             5901 non-null   str    
 11  Region            5901 non-null   str    
 12  Product ID        5901 non-null   str    
 13  Category          5901 non-null   str    
 14  Sub-Category      5901 non-null   str    
 15  Product Name      5901 non-null   str    
 16  Sales             5901 non-null   float64
 17  Quanti

In [18]:
pip install plotly

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
from dash import Dash, html, dcc, Input, Output
import dash_ag_grid as dag
import plotly.express as px

app = Dash()

df = pd.read_csv('../data/superstore.csv')
# Convert 'Order Date' to datetime
df['Order Date'] = pd.to_datetime(df['Order Date'], format='mixed')
df['Month'] = df['Order Date'].dt.to_period('M')

# Requires Dash 2.17.0 or later
app.layout = [
    # Header
    html.H1("Superstore Sales Dashboard", style={'textAlign': 'center'}),
    # Filters
    html.Div([
    dcc.Dropdown(
        id='region-filter',
        options=[{'label': r, 'value': r} for r in df['Region'].unique()],
        value=df['Region'].unique()[0]
    ),
    dcc.Dropdown(
        id = 'category-filter',
        options=[{'label': c, 'value': c} for c in df['Category'].unique()],
        value=df['Category'].unique()[0]
    ),
    dcc.DatePickerRange(
        id='date-picker',
        start_date=df['Order Date'].min(),
        end_date=df['Order Date'].max(),
        display_format='YYYY-MM-DD'
    ),
    html.Div(id='kpi-cards', style={'display': 'flex', 'gap':'20px'}),
    ], style={'display': 'flex', 'gap':'20px'},),
    
    html.Br(),
    # KPI Cards
    #html.Div(id='kpi-cards', style={'display': 'flex', 'gap':'20px'}),

    html.Br(),

    # Additional charts or tables can be added here
    html.Div([
        dcc.Graph(id='sales-trend'),
        dcc.Graph(id='category-pie')
    ], style={'display': 'flex', 'gap':'20px'}),

    html.Div([
        dcc.Graph(id='subcat-bar'),
        dcc.Graph(id='profit-region')
    ], style={'display': 'flex', 'gap':'20px'}),

    html.Br(),

    html.H3("⚠️ Loss-Making Products"),
    html.Div(id='loss-table')

]
# callback
@app.callback(
    [Output('kpi-cards', 'children'),
    Output('sales-trend', 'figure'),
    Output('category-pie', 'figure'),
    Output('subcat-bar', 'figure'),
    Output('profit-region', 'figure'),
    Output('loss-table', 'children')],
    
    [Input('region-filter', 'value'),
    Input('category-filter', 'value'),
    Input('date-picker', 'start_date'),
    Input('date-picker', 'end_date')]
)
def update_graph(selected_region, selected_category, start_date, end_date):
    # filter the DataFrame based on the selected region
    filtered_df = df.copy()
    
    if selected_region:
        filtered_df = filtered_df[filtered_df['Region'] == selected_region] 
    if selected_category:
        filtered_df = filtered_df[filtered_df['Category'] == selected_category]
    
    filtered_df = filtered_df[
        (filtered_df['Order Date'] >= start_date) &
        (filtered_df['Order Date'] <= end_date)
    ]

    # KPIs
    total_sales = filtered_df['Sales'].sum()
    total_profit = filtered_df['Profit'].sum()
    total_orders = filtered_df.shape[0]
    profit_ratio = round(total_profit / total_sales * 100, 2) if total_sales != 0 else 0

    kpis = [
        html.Div(f"💰 Sales: ${total_sales}", style=card_style()),
        html.Div(f"📈 Profit: ${total_profit}", style=card_style()),
        html.Div(f"🛒 Orders: {total_orders}", style=card_style()),
        html.Div(f"📊 Profit Ratio: {profit_ratio}%", style=card_style())
    ]    
    
    
    
    
    fig = px.line(filtered_df, x='Order Date', y='Sales', title=f'Sales Over Time - {selected_region}')
    return fig




# Card Style Function
def card_style():
    return {
        'padding': '20px',
        'border': '1px solid #ddd',
        'borderRadius': '10px',
        'boxShadow': '2px 2px 5px lightgrey',
        'fontSize': '20px'
    }

if __name__ == '__main__':
    app.run(debug=True)


[2026-04-15 10:55:20,902] ERROR in app: Exception on /_dash-update-component [POST]
Traceback (most recent call last):
  File "C:\Users\mukta\OneDrive\Documents\JobHunts\Data-Science-Projects\.venv\Lib\site-packages\pandas\core\indexes\base.py", line 3641, in get_loc
    return self._engine.get_loc(casted_key)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "pandas/_libs/index.pyx", line 168, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/index.pyx", line 197, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/hashtable_class_helper.pxi", line 7668, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas/_libs/hashtable_class_helper.pxi", line 7676, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: 'Profit'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "C:\Users\mukta\OneDrive\Documents\JobHunts\Data-Science-Projects\.venv\Lib\site-packages\flask\app.py", line 917,